# Binge-Scrolling Behavior & Academic Impact Analysis

**Dataset:** Survey of 150 university students on social media usage habits and 24 Likert-scale (1–5) items measuring compulsive scrolling behavior and academic impact.

**Goals of this notebook:**
1. Build and validate two composite psychometric scales (Binge-Scrolling Index, Academic Impact Index)
2. Test the correlation between binge-scrolling and academic disruption
3. Compare scrolling severity across demographics (academic year, gender, daily hours, etc.)
4. Segment students into behavioral personas using K-means clustering
5. Visualize all findings

> Run this notebook in Google Colab — just upload `Cleaned_Thesis_Data.csv` when prompted in the cell below, or place it in the same folder as this notebook if running locally.


## 1. Setup

In [ ]:
# Install/import required libraries (Colab has most of these pre-installed)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, f_oneway, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110


## 2. Load the data

**If running in Google Colab**, run the cell below to upload `Cleaned_Thesis_Data.csv` from your computer.
**If running locally / the file is already in the repo**, skip the upload cell and just run the `pd.read_excel(...)` line.

In [ ]:
# --- Colab file upload (skip this cell if running locally with the file already present) ---
try:
    from google.colab import files
    uploaded = files.upload()
    FILENAME = list(uploaded.keys())[0]
except ImportError:
    FILENAME = "Cleaned_Thesis_Data.csv"  # local path


In [ ]:
df = pd.read_csv(FILENAME)
print(df.shape)
df.head()


## 3. Define the item groups

- **Binge-Scrolling items (Q13–Q24):** compulsive use behaviors and emotional aftermath (guilt, depression, loss of control)
- **Academic Impact items (Q25–Q28):** procrastination, study efficiency, concentration, anxiety without social media

In [ ]:
BINGE_COLS = [
    "Q13_Continuous_Scroll", "Q14_Scroll_Until_Bored", "Q15_Scroll_No_Purpose",
    "Q16_Scroll_When_Alone", "Q17_Physically_Uncomfortable", "Q18_Feel_Guilty",
    "Q19_Feel_Depressed", "Q20_Mad_At_Self", "Q21_Struggle_To_Control",
    "Q22_Thinking_About_Scrolling", "Q23_Scroll_More_Than_Planned", "Q24_Cannot_Stop_Self",
]

ACADEMIC_COLS = [
    "Q25_Procrastinate_Academic_Work", "Q26_Reduces_Study_Efficiency",
    "Q27_Lose_Study_Concentration", "Q28_Anxious_Without_Social_Media",
]


## 4. Build composite indices & check reliability

Before averaging Likert items into a single score, we check **Cronbach's alpha** — a measure of internal consistency. Above 0.7 is generally considered acceptable, above 0.8 is good.

In [ ]:
def cronbach_alpha(df_sub: pd.DataFrame) -> float:
    """Standard Cronbach's alpha for a set of Likert items."""
    df_sub = df_sub.dropna()
    item_vars = df_sub.var(axis=0, ddof=1)
    total_var = df_sub.sum(axis=1).var(ddof=1)
    n_items = df_sub.shape[1]
    return (n_items / (n_items - 1)) * (1 - item_vars.sum() / total_var)

alpha_binge = cronbach_alpha(df[BINGE_COLS])
alpha_academic = cronbach_alpha(df[ACADEMIC_COLS])

print(f"Cronbach's alpha — Binge-Scrolling Index:  {alpha_binge:.3f}")
print(f"Cronbach's alpha — Academic Impact Index:  {alpha_academic:.3f}")


In [ ]:
df["Binge_Index"] = df[BINGE_COLS].mean(axis=1)
df["Academic_Impact_Index"] = df[ACADEMIC_COLS].mean(axis=1)

df[["Binge_Index", "Academic_Impact_Index"]].describe()


## 5. Correlation between binge-scrolling and academic impact

In [ ]:
r, p = pearsonr(df["Binge_Index"], df["Academic_Impact_Index"])
print(f"Pearson r = {r:.3f}, p = {p:.5f}")

plt.figure(figsize=(7, 5))
sns.regplot(
    data=df, x="Binge_Index", y="Academic_Impact_Index",
    scatter_kws={"alpha": 0.6, "s": 40}, line_kws={"color": "crimson"}
)
plt.title(f"Binge-Scrolling vs Academic Impact  (r={r:.2f}, p<{max(p,0.0001):.4f})")
plt.xlabel("Binge-Scrolling Index")
plt.ylabel("Academic Impact Index")
plt.tight_layout()
plt.show()


## 6. Group comparisons

Testing whether binge-scrolling severity differs across demographics and usage habits using one-way ANOVA (for multi-category variables) and t-tests (for binary variables).

In [ ]:
print("One-way ANOVA — Binge_Index across groups:\n")
for col in ["Field_of_Study", "Age", "Academic_Year", "Binge_Scroll_Platform",
            "Peak_Usage_Time", "Daily_Hours_Spent"]:
    groups = [g["Binge_Index"].values for _, g in df.groupby(col)]
    f_stat, p_val = f_oneway(*groups)
    flag = "**significant**" if p_val < 0.05 else ""
    print(f"  {col:25s} F={f_stat:6.2f}   p={p_val:.4f}  {flag}")


In [ ]:
male = df.loc[df["Gender"] == "Male", "Binge_Index"]
female = df.loc[df["Gender"] == "female", "Binge_Index"]
t_stat, p_val = ttest_ind(male, female)
print(f"Gender (Male vs female) t-test: t={t_stat:.2f}, p={p_val:.4f}")
print(f"  Male mean:   {male.mean():.2f}")
print(f"  Female mean: {female.mean():.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

order_year = ["1st Year", "2nd Year", "3rd Year", "4th Year", "Masters"]
sns.barplot(data=df, x="Academic_Year", y="Binge_Index", order=order_year, ax=axes[0])
axes[0].set_title("By Academic Year (p < .001)")
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=df, x="Gender", y="Binge_Index", ax=axes[1])
axes[1].set_title("By Gender (p = .003)")

order_hours = ["1–2 hours", "2–3 hours", "3–4 hours", "More than 4 hours"]
sns.barplot(data=df, x="Daily_Hours_Spent", y="Binge_Index", order=order_hours, ax=axes[2])
axes[2].set_title("By Daily Hours Online (p = .118)")
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 7. Behavioral personas via K-means clustering

Clustering all 16 Likert items (12 binge-scrolling + 4 academic impact) to segment students into behavioral profiles.

In [ ]:
features = df[BINGE_COLS + ACADEMIC_COLS]
X_scaled = StandardScaler().fit_transform(features)

print("Silhouette scores by k:")
for k in (2, 3, 4):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    print(f"  k={k}: {silhouette_score(X_scaled, labels):.3f}")


In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
df["Cluster"] = km.labels_

# Order clusters by severity and label them
order = df.groupby("Cluster")["Binge_Index"].mean().sort_values().index.tolist()
label_map = {order[0]: "Low-Risk", order[1]: "Moderate", order[2]: "At-Risk"}
df["Persona"] = df["Cluster"].map(label_map)

persona_summary = df.groupby("Persona")[["Binge_Index", "Academic_Impact_Index"]].agg(["mean", "count"])
persona_summary


In [ ]:
plt.figure(figsize=(7, 5))
palette = {"Low-Risk": "#3DDC97", "Moderate": "#FFC857", "At-Risk": "#FF4D6D"}
sns.scatterplot(
    data=df, x="Binge_Index", y="Academic_Impact_Index",
    hue="Persona", palette=palette, s=60, alpha=0.8,
    hue_order=["Low-Risk", "Moderate", "At-Risk"]
)
plt.title("Behavioral Personas: Binge-Scrolling vs Academic Impact")
plt.xlabel("Binge-Scrolling Index")
plt.ylabel("Academic Impact Index")
plt.tight_layout()
plt.show()

print(df["Persona"].value_counts(normalize=True).mul(100).round(1))


## 8. Item-level breakdown

In [ ]:
item_means = df[BINGE_COLS].mean().sort_values(ascending=False)
item_means.index = [c.split("_", 1)[1].replace("_", " ") for c in item_means.index]

plt.figure(figsize=(8, 6))
sns.barplot(x=item_means.values, y=item_means.index, color="#FF4D6D")
plt.xlabel("Mean agreement (1–5)")
plt.title("Binge-Scrolling Item Means")
plt.xlim(0, 5)
plt.tight_layout()
plt.show()


## 9. Save processed data

Exports a clean CSV with the composite indices and persona labels attached, ready for further analysis or for feeding into a dashboard.

In [ ]:
df.to_csv("processed_data.csv", index=False)
print("Saved processed_data.csv")
df.head()


## Summary of key findings

- **Reliability:** Both composite scales show strong internal consistency (Binge-Scrolling α = 0.885, Academic Impact α = 0.814)
- **Correlation:** Binge-scrolling severity correlates strongly with academic impact (r = 0.78, p < .001)
- **Group differences:** Academic year (p < .001) and gender (p = .003) show significant differences in scrolling severity; daily hours online trends upward but isn't statistically significant
- **Personas:** Students cluster into three behavioral profiles — Low-Risk (21%), Moderate (35%), and At-Risk (44%) — with binge-scrolling and academic impact rising together across the groups
